# ZX Calculus: A Graphical Language for Quantum Circuits
## Grounded in the Ising-Trotter and Surface Code Pipeline

This notebook is the fourth part of the series. It is designed to be read alongside the three prior works:

| Notebook | What it contributes here |
|---|---|
| `cliff_trott_ising.ipynb` | Ising Hamiltonian, Trotterization, Cliffordization, Pauli twirling |
| `7_surface_code_tutorial.ipynb` | Rotated surface code, stabilizers, MWPM decoding, lattice surgery |
| `ising_surface_code_qec.ipynb` | End-to-end QEC pipeline bridging both worlds via Cliffordization |

### What ZX calculus adds

ZX calculus is a **graphical rewriting system** for quantum circuits. Instead of matrix multiplication, you manipulate diagrams using a small set of rules. It turns out that:

- Every Clifford circuit is a ZX diagram — so everything Cliffordized in the prior notebooks can be drawn and simplified graphically.
- Surface code stabilizers are **spiders** — the fundamental ZX generators.
- Pauli twirling, Cliffordization, and lattice surgery all have clean ZX-diagram interpretations.
- ZX rewriting is the backbone of modern circuit optimisation and QEC formal verification tools.

### Notebook structure

1. Installation and imports
2. The two generators: Z and X spiders
3. The five rewrite rules
4. Clifford circuits as ZX diagrams
5. The Ising Trotter step in ZX
6. Surface code stabilizers as spiders
7. Syndrome extraction circuits in ZX
8. Pauli propagation and Cliffordization via ZX
9. Lattice surgery as ZX composition
10. Discussion and further reading

---
**Prerequisites:** Sections 1–4 of `7_surface_code_tutorial.ipynb` and familiarity with Clifford circuits from `ising_surface_code_qec.ipynb`.

---
## 1. Installation and Imports

```bash
pip install qiskit qiskit-aer stim pymatching matplotlib numpy networkx
```

ZX diagrams in this notebook are drawn using plain `matplotlib` — no additional ZX library is required. The circuits throughout use exactly the same parameters as `ising_surface_code_qec.ipynb` so results are directly comparable.

In [ ]:
!pip install qiskit qiskit_aer stim pymatching matplotlib numpy networkx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Circle, FancyBboxPatch
import warnings
warnings.filterwarnings("ignore")

# Qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import random_clifford
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

# Stim / PyMatching
import stim
import pymatching

print("All imports successful.")
print(f"  stim      : {stim.__version__}")
print(f"  pymatching: {pymatching.__version__}")

# ── Global parameters (identical to ising_surface_code_qec.ipynb) ──────────
N_QUBITS      = 4
J             = 0.2
H_FIELD       = 1.2
ALPHA         = np.pi / 8
TOTAL_TIME    = 1.0
P_PHYS        = 1e-2
CODE_DIST     = 3
QEC_ROUNDS    = CODE_DIST
NUM_SHOTS     = 20_000

print("\nParameters loaded (identical to ising_surface_code_qec.ipynb).")

**Reading the import block.**  
The same two ecosystems from `ising_surface_code_qec.ipynb` are loaded here: Qiskit for circuit construction and Aer simulation, and stim/pymatching for stabilizer-level QEC. The global parameters are deliberately identical — every circuit, every logical error rate, and every resource estimate produced in this notebook is directly comparable to results in the prior notebooks.

The ZX calculus machinery (drawing, rewriting) is implemented from scratch with matplotlib so that each diagram is fully transparent. In production code you would use the `pyzx` library, which automates spider fusion, the `full_reduce` simplification algorithm, and export to various formats.

---
## 2. The Two Generators: Z and X Spiders

ZX calculus has exactly two generators, called **spiders**, from which every quantum computation is built.

### 2.1 Z spider

A Z spider with $n$ input wires, $m$ output wires, and phase angle $\alpha \in [0, 2\pi)$ denotes the linear map:

$$Z(\alpha)_{n,m} = |0\rangle^{\otimes m}\langle 0|^{\otimes n} + e^{i\alpha}|1\rangle^{\otimes m}\langle 1|^{\otimes n}$$

Drawn as a **green circle** labelled $\alpha$ (or unlabelled when $\alpha = 0$).

### 2.2 X spider

An X spider with $n$ inputs, $m$ outputs, and phase $\alpha$ denotes:

$$X(\alpha)_{n,m} = |+\rangle^{\otimes m}\langle +|^{\otimes n} + e^{i\alpha}|-\rangle^{\otimes m}\langle -|^{\otimes n}$$

Drawn as a **red circle** labelled $\alpha$.

### 2.3 Connection to gates you already know

| Gate | ZX description |
|---|---|
| $Z$ gate | Z spider, $n=m=1$, $\alpha = \pi$ |
| $X$ gate | X spider, $n=m=1$, $\alpha = \pi$ |
| $S$ gate | Z spider, $n=m=1$, $\alpha = \pi/2$ |
| $R_z(\theta)$ | Z spider, $\alpha = \theta$ |
| $R_x(\theta)$ | X spider, $\alpha = \theta$ |
| $H$ gate | Special: connects Z and X basis (Hadamard box) |
| CNOT | Composed: Z spider (control) fused to X spider (target) |

The $|0\rangle$ ket is a Z spider with 0 inputs and 1 output at $\alpha=0$. The $\langle +|$ bra is an X spider with 1 input and 0 outputs at $\alpha=0$.

In [ ]:
def draw_spider(ax, x, y, color, alpha_label='', radius=0.28,
                n_inputs=1, n_outputs=1, wire_len=0.55, label_offset=0.45):
    """Draw a single ZX spider with its wires.
    color: '#4CAF50' (green/Z) or '#EF5350' (red/X)
    """
    # Wires
    for i in range(n_inputs):
        yw = y + (i - (n_inputs - 1) / 2) * 0.35
        ax.annotate('', xy=(x - radius, yw), xytext=(x - radius - wire_len, yw),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    for i in range(n_outputs):
        yw = y + (i - (n_outputs - 1) / 2) * 0.35
        ax.annotate('', xy=(x + radius + wire_len, yw), xytext=(x + radius, yw),
                    arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    # Circle
    circle = Circle((x, y), radius, color=color, ec='black', lw=1.8, zorder=5)
    ax.add_patch(circle)
    # Phase label
    if alpha_label:
        ax.text(x, y, alpha_label, ha='center', va='center',
                fontsize=9, fontweight='bold', color='white', zorder=6)


def draw_hadamard(ax, x, y, wire_len=0.45):
    """Draw an H (Hadamard) box connecting Z and X colour."""
    ax.annotate('', xy=(x - 0.2, y), xytext=(x - 0.2 - wire_len, y),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    ax.annotate('', xy=(x + 0.2 + wire_len, y), xytext=(x + 0.2, y),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    box = FancyBboxPatch((x - 0.2, y - 0.2), 0.4, 0.4,
                          boxstyle='round,pad=0.03', facecolor='#FFEB3B',
                          edgecolor='black', lw=1.8, zorder=5)
    ax.add_patch(box)
    ax.text(x, y, 'H', ha='center', va='center',
            fontsize=10, fontweight='bold', color='black', zorder=6)


# ── Figure: basic generators ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(13, 2.8))
Z_COLOR = '#4CAF50'
X_COLOR = '#EF5350'

for ax in axes:
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.8, 0.8)
    ax.set_aspect('equal'); ax.axis('off')

# 1. Z spider (phase = 0)
draw_spider(axes[0], 0, 0, Z_COLOR, alpha_label='0')
axes[0].set_title('Z spider  (α=0)', fontsize=10)

# 2. Z spider (phase = α)
draw_spider(axes[1], 0, 0, Z_COLOR, alpha_label='α')
axes[1].set_title('Z spider  (phase α)', fontsize=10)

# 3. X spider (phase = 0)
draw_spider(axes[2], 0, 0, X_COLOR, alpha_label='0')
axes[2].set_title('X spider  (α=0)', fontsize=10)

# 4. Hadamard box
draw_hadamard(axes[3], 0, 0)
axes[3].set_title('Hadamard box', fontsize=10)

fig.suptitle('ZX Calculus Generators', fontsize=12, fontweight='500', y=1.01)
plt.tight_layout()
plt.show()

print("Z_COLOR = green (#4CAF50),  X_COLOR = red (#EF5350)")

**Reading the generators figure.**  
The four panels show everything you need to build any quantum computation in ZX calculus:

- *Green spiders (Z-type)* are diagonal in the computational basis $\{|0\rangle, |1\rangle\}$. A Z spider with $\alpha = 0$ acts as a "copy" map in the Z basis and as an addition map in the X basis. Adding phase $\alpha$ introduces a relative phase between the $|0\rangle^{\otimes}$ and $|1\rangle^{\otimes}$ branches.
- *Red spiders (X-type)* are identical in structure but diagonal in the Hadamard-rotated basis $\{|+\rangle, |-\rangle\}$. They are copy maps in the X basis.
- *Hadamard boxes* convert between Z and X colour. Placing an H box between a green wire and a red spider is equivalent to the standard Hadamard gate. Importantly, H boxes can be "passed through" spiders using the colour-change rule (Section 3.5).

Wires represent qubits. The direction of time flows left to right: inputs are on the left, outputs on the right. A wire with zero inputs (a "cap") prepares a state; a wire with zero outputs (a "cup") performs a post-selection or measurement.

---
## 3. The Five Rewrite Rules

ZX calculus is **sound and complete** for Clifford+T circuits: two diagrams that compute the same linear map can always be connected by a finite sequence of these five rules (plus their mirror images and colour-swapped versions).

### Rule 1 — Spider Fusion (S1)
Two same-colour spiders connected by a wire fuse into one, with phases added:
$$Z(\alpha) \circ Z(\beta) \;=\; Z(\alpha + \beta)$$

### Rule 2 — Identity Removal (S2)
A spider with one input, one output, and phase 0 is the identity wire:
$$Z(0)_{1,1} \;=\; \mathbb{1}$$

### Rule 3 — Bialgebra / Copy Rule (B)
A Z spider and an X spider interact as a classical copying+adding structure:
$$\text{(Z copy)} \circ \text{(X add)} \;=\; \text{(X add)} \circ \text{(Z copy)} \otimes \text{(Z copy)}$$
This is the **entanglement rule** — it encodes how CX gates propagate Pauli errors.

### Rule 4 — Euler Decomposition / π-copy (π)
A phase-π spider of one colour passes through an opposite-colour spider, picking up a phase:
$$X(\pi) \circ Z(\alpha) \;=\; Z(-\alpha) \circ X(\pi)$$
Equivalently: flipping an X through a Z gate conjugates the phase — this is Pauli propagation.

### Rule 5 — Hopf Rule / Colour Change (H)
Adding an H box on every wire of a spider converts it to the opposite colour:
$$H \circ Z(\alpha) \circ H \;=\; X(\alpha)$$
This is the Hadamard conjugation relation $H Z H = X$, $H X H = Z$.

In [ ]:
def draw_wire(ax, x0, x1, y, color='black', lw=1.8):
    ax.plot([x0, x1], [y, y], color=color, lw=lw, zorder=2)

def spider(ax, x, y, color, phase='', r=0.22):
    c = Circle((x, y), r, color=color, ec='black', lw=1.6, zorder=5)
    ax.add_patch(c)
    if phase:
        ax.text(x, y, phase, ha='center', va='center',
                fontsize=8, fontweight='bold', color='white', zorder=6)

def equals_sign(ax, x, y):
    ax.text(x, y, '=', ha='center', va='center', fontsize=16, color='#333')

fig, axes = plt.subplots(5, 1, figsize=(11, 10))
Z_COLOR = '#4CAF50'
X_COLOR = '#EF5350'
H_COLOR = '#FFEB3B'

for ax in axes:
    ax.set_aspect('equal')
    ax.axis('off')

# ── Rule 1: Spider fusion ────────────────────────────────────────────────────
ax = axes[0]; ax.set_xlim(0, 11); ax.set_ylim(-0.6, 0.6)
ax.set_title('Rule 1 — Spider Fusion: same-colour spiders connected by a wire fuse; phases add',
             fontsize=9, loc='left')
# LHS
draw_wire(ax, 0.2, 1.0, 0); spider(ax, 1.2, 0, Z_COLOR, 'α')
draw_wire(ax, 1.45, 2.4, 0); spider(ax, 2.6, 0, Z_COLOR, 'β')
draw_wire(ax, 2.85, 3.5, 0)
# Equals
equals_sign(ax, 4.3, 0)
# RHS
draw_wire(ax, 5.1, 5.8, 0); spider(ax, 6.0, 0, Z_COLOR, 'α+β')
draw_wire(ax, 6.25, 7.0, 0)

# ── Rule 2: Identity removal ─────────────────────────────────────────────────
ax = axes[1]; ax.set_xlim(0, 11); ax.set_ylim(-0.6, 0.6)
ax.set_title('Rule 2 — Identity Removal: phase-0 spider with 1 input, 1 output = identity wire',
             fontsize=9, loc='left')
draw_wire(ax, 0.2, 1.0, 0); spider(ax, 1.2, 0, Z_COLOR, '0')
draw_wire(ax, 1.45, 3.0, 0)
equals_sign(ax, 4.3, 0)
draw_wire(ax, 5.1, 7.5, 0)

# ── Rule 3: π-copy / Pauli propagation ──────────────────────────────────────
ax = axes[2]; ax.set_xlim(0, 11); ax.set_ylim(-0.8, 0.8)
ax.set_title('Rule 4 — π-copy (Pauli Propagation): X(π) passes through Z(α), negating the phase',
             fontsize=9, loc='left')
# LHS: X(π) -- wire -- Z(α)
draw_wire(ax, 0.2, 1.0, 0); spider(ax, 1.2, 0, X_COLOR, 'π')
draw_wire(ax, 1.45, 2.4, 0); spider(ax, 2.6, 0, Z_COLOR, 'α')
draw_wire(ax, 2.85, 3.5, 0)
equals_sign(ax, 4.3, 0)
# RHS: Z(-α) -- wire -- X(π)
draw_wire(ax, 5.1, 5.8, 0); spider(ax, 6.0, 0, Z_COLOR, '-α')
draw_wire(ax, 6.25, 7.2, 0); spider(ax, 7.4, 0, X_COLOR, 'π')
draw_wire(ax, 7.65, 8.3, 0)

# ── Rule 4: Colour change via H ──────────────────────────────────────────────
ax = axes[3]; ax.set_xlim(0, 11); ax.set_ylim(-0.6, 0.6)
ax.set_title('Rule 5 — Colour Change: H conjugation converts Z spider to X spider',
             fontsize=9, loc='left')
# LHS: H -- Z(α) -- H
draw_wire(ax, 0.2, 0.7, 0)
hbox1 = FancyBboxPatch((0.7, -0.18), 0.36, 0.36, boxstyle='round,pad=0.02',
                         facecolor=H_COLOR, edgecolor='black', lw=1.5, zorder=5)
ax.add_patch(hbox1); ax.text(0.88, 0, 'H', ha='center', va='center', fontsize=8, fontweight='bold')
draw_wire(ax, 1.08, 1.8, 0); spider(ax, 2.0, 0, Z_COLOR, 'α')
draw_wire(ax, 2.25, 2.9, 0)
hbox2 = FancyBboxPatch((2.9, -0.18), 0.36, 0.36, boxstyle='round,pad=0.02',
                         facecolor=H_COLOR, edgecolor='black', lw=1.5, zorder=5)
ax.add_patch(hbox2); ax.text(3.08, 0, 'H', ha='center', va='center', fontsize=8, fontweight='bold')
draw_wire(ax, 3.28, 3.8, 0)
equals_sign(ax, 4.5, 0)
# RHS: X(α)
draw_wire(ax, 5.3, 6.0, 0); spider(ax, 6.2, 0, X_COLOR, 'α')
draw_wire(ax, 6.45, 7.2, 0)

# ── Rule 5: Bialgebra (copy + add) ──────────────────────────────────────────
ax = axes[4]; ax.set_xlim(0, 11); ax.set_ylim(-1.0, 1.0)
ax.set_title('Rule 3 — Bialgebra: Z-copy through X-add (key rule for CNOT/entanglement)',
             fontsize=9, loc='left')
# LHS: Z(0) 1→2 -- two X(0) 2→1
draw_wire(ax, 0.2, 0.8, 0)
spider(ax, 1.0, 0, Z_COLOR, '')
# Two outputs
draw_wire(ax, 1.22, 2.0,  0.3); spider(ax, 2.2, 0.3, X_COLOR, '')
draw_wire(ax, 1.22, 2.0, -0.3); spider(ax, 2.2,-0.3, X_COLOR, '')
draw_wire(ax, 2.42,  3.0,  0.3)
draw_wire(ax, 2.42,  3.0, -0.3)
equals_sign(ax, 4.0, 0)
# RHS — two Z copies feeding one X add
draw_wire(ax, 5.0, 5.7,  0.3); spider(ax, 5.9, 0.3, Z_COLOR, '')
draw_wire(ax, 5.0, 5.7, -0.3); spider(ax, 5.9,-0.3, Z_COLOR, '')
draw_wire(ax, 6.12, 7.0,  0.3)
draw_wire(ax, 6.12, 7.0, -0.3)
draw_wire(ax, 6.12, 6.8,  0.3); ax.plot([6.8, 7.0], [0.3, 0], 'k-', lw=1.8)
draw_wire(ax, 6.12, 6.8, -0.3); ax.plot([6.8, 7.0], [-0.3, 0], 'k-', lw=1.8)
spider(ax, 7.2, 0, X_COLOR, '')
draw_wire(ax, 7.42, 8.2, 0)

fig.suptitle('The Five ZX Rewrite Rules', fontsize=13, fontweight='500', y=1.0)
plt.tight_layout(h_pad=0.8)
plt.show()

**Reading the rewrite rules.**  
Each rule is an equation between ZX diagrams. Their power comes from the fact that they can be applied in any order, to any sub-diagram, regardless of the surrounding context.

*Spider Fusion (Rule 1)* is the workhorse of circuit simplification. In the Clifford proxy circuits from `ising_surface_code_qec.ipynb`, adjacent single-qubit Clifford gates on the same qubit fuse immediately: two Z spiders of phases $\pi/2$ and $\pi/2$ fuse into a single Z spider of phase $\pi$ (the Z gate). This is exactly what the Qiskit transpiler's `optimization_level=1` does, but stated graphically.

*Identity Removal (Rule 2)* removes zero-phase spiders. Any $R_z(0)$ or $R_x(0)$ gate (identity rotation) is simply deleted from the diagram without changing the linear map.

*π-copy / Pauli Propagation (Rule 4)* is the ZX encoding of the Pauli commutation relations. When you apply an X error before a $R_z(\alpha)$ gate, the error emerges on the other side as an X error but the $R_z$ gate acquires a phase flip — exactly what `apply_pauli_twirl()` exploits. Pauli twirling, viewed graphically, is the process of using Rule 4 to propagate all Pauli errors to the same side of each CX gate.

*Colour Change (Rule 5)* captures $HZH = X$ graphically. A green spider conjugated by H boxes on every wire becomes a red spider with the same phase. This is why Z-basis stabilizers become X-basis stabilizers after Hadamard rotation.

*Bialgebra (Rule 3)* is the deepest rule: it describes how copying and adding interact across colour boundaries, capturing the full CX entanglement structure. It is the rule you need to verify that the CNOT gate correctly propagates Z errors on the control and X errors on the target — the key fact behind surface code stabilizer measurements.

---
## 4. Clifford Circuits as ZX Diagrams

Every gate in the Clifford basis `{H, S, S†, CX, X, Y, Z}` used throughout the prior notebooks is a ZX diagram. This section builds the dictionary.

### 4.1 Single-qubit Clifford gates

| Gate | ZX representation |
|---|---|
| $I$ | Plain wire (Rule 2) |
| $Z$ | Z spider, $\alpha = \pi$ |
| $X$ | X spider, $\alpha = \pi$ |
| $Y$ | Z spider ($\pi$) fused with X spider ($\pi$) |
| $S$ | Z spider, $\alpha = \pi/2$ |
| $S^\dagger$ | Z spider, $\alpha = -\pi/2 = 3\pi/2$ |
| $H$ | Hadamard box (or: Z($\pi/2$) — X($\pi/2$) — Z($\pi/2$) by Euler decomposition) |
| $\sqrt{X}$ (`sx`) | X spider, $\alpha = \pi/2$ |

### 4.2 Two-qubit Clifford gate: CNOT

The CNOT (CX) gate is the most important gate in both the Ising Trotter circuit and the surface code syndrome extraction circuit. In ZX calculus:

$$\mathrm{CX} = \text{Z spider (control, 1 in 2 out)} \times_\text{wire} \text{X spider (target, 2 in 1 out)}$$

The control qubit has a Z spider that copies: it sends one copy to the target qubit and one copy to itself. The target qubit has an X spider that adds: it XORs the copy from the control into the target.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
Z_COLOR = '#4CAF50'
X_COLOR = '#EF5350'
H_COLOR = '#FFEB3B'

for ax in axes:
    ax.set_aspect('equal'); ax.axis('off')

# ── Panel A: CNOT as ZX diagram ───────────────────────────────────────────
ax = axes[0]; ax.set_xlim(-0.3, 3.8); ax.set_ylim(-1.4, 1.2)
ax.set_title('CNOT as ZX diagram', fontsize=10)

# Control wire → Z spider → wire → output
ax.plot([-0.2, 0.8],  [0.6, 0.6], 'k-', lw=2, zorder=2)
cz = Circle((1.0, 0.6), 0.22, color=Z_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(cz)
ax.plot([1.22, 3.5], [0.6, 0.6], 'k-', lw=2, zorder=2)  # output

# Connecting wire from Z spider to X spider
ax.plot([1.0, 1.0], [0.38, -0.58], 'k-', lw=2, zorder=2)

# Target wire → X spider → wire → output
ax.plot([-0.2, 0.78], [-0.8, -0.8], 'k-', lw=2, zorder=2)
cx_s = Circle((1.0, -0.8), 0.22, color=X_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(cx_s)
ax.plot([1.22, 3.5], [-0.8, -0.8], 'k-', lw=2, zorder=2)

ax.text(-0.25, 0.6,  'ctrl', ha='right', fontsize=8, color='#333')
ax.text(-0.25, -0.8, 'tgt',  ha='right', fontsize=8, color='#333')
ax.text(3.55, 0.6,   'ctrl', ha='left',  fontsize=8, color='#333')
ax.text(3.55, -0.8,  'tgt',  ha='left',  fontsize=8, color='#333')

# ── Panel B: Z error propagation through CNOT ─────────────────────────────
ax = axes[1]; ax.set_xlim(-0.3, 5.0); ax.set_ylim(-1.4, 1.2)
ax.set_title('Z error on control propagates to target', fontsize=10)

# Z(π) error before CNOT
ax.plot([-0.2, 0.4],  [0.6, 0.6], 'k-', lw=2)
ez = Circle((0.6, 0.6), 0.18, color=Z_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(ez)
ax.text(0.6, 0.6, 'π', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
ax.plot([0.78, 1.5],  [0.6, 0.6], 'k-', lw=2)

# CNOT
cz2 = Circle((1.7, 0.6), 0.2, color=Z_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(cz2)
ax.plot([1.7, 1.7], [0.4, -0.58], 'k-', lw=2)
ax.plot([-0.2, 1.5], [-0.8, -0.8], 'k-', lw=2)
cx2 = Circle((1.7, -0.8), 0.2, color=X_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(cx2)

ax.text(2.8, 0, '= (by Rule 3)', ha='center', fontsize=8, style='italic', color='#555')

# RHS: Z errors on both wires after CNOT
ax.plot([1.9, 3.2], [0.6, 0.6], 'k-', lw=2)
ez2 = Circle((3.4, 0.6), 0.18, color=Z_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(ez2)
ax.text(3.4, 0.6, 'π', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
ax.plot([3.58, 4.8], [0.6, 0.6], 'k-', lw=2)

ax.plot([1.9, 3.2], [-0.8, -0.8], 'k-', lw=2)
ez3 = Circle((3.4, -0.8), 0.18, color=Z_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(ez3)
ax.text(3.4, -0.8, 'π', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
ax.plot([3.58, 4.8], [-0.8, -0.8], 'k-', lw=2)

# ── Panel C: Clifford basis gate recap ────────────────────────────────────
ax = axes[2]; ax.set_xlim(0, 5); ax.set_ylim(-0.5, 5.5)
ax.set_title('Clifford basis → ZX phases', fontsize=10)
gate_data = [
    ('I',       Z_COLOR,  '0',    0.0),
    ('Z',       Z_COLOR,  'π',    1.0),
    ('S',       Z_COLOR,  'π/2',  2.0),
    ('S†',      Z_COLOR,  '3π/2', 3.0),
    ('X',       X_COLOR,  'π',    4.0),
    ('√X (sx)', X_COLOR,  'π/2',  5.0),
]
for label, col, phase, ypos in gate_data:
    ax.plot([0.5, 1.1], [ypos, ypos], 'k-', lw=1.8)
    c = Circle((1.3, ypos), 0.2, color=col, ec='black', lw=1.5, zorder=5)
    ax.add_patch(c)
    ax.text(1.3, ypos, phase, ha='center', va='center', fontsize=6.5,
            color='white', fontweight='bold', zorder=6)
    ax.plot([1.5, 2.2], [ypos, ypos], 'k-', lw=1.8)
    ax.text(2.4, ypos, f'← {label}', fontsize=9, va='center', color='#222')
ax.axis('off')

plt.suptitle('CNOT in ZX and Error Propagation Rules', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**Reading the CNOT and error propagation figure.**  

*Left panel (CNOT as ZX diagram):* The CNOT gate is a Z spider on the control qubit wire connected by a vertical "internal" wire to an X spider on the target qubit wire. The Z spider copies the computational-basis state from the control; the X spider adds it to the target. This is the ZX realisation of the CX unitary $|00\rangle\langle 00| + |01\rangle\langle 01| + |11\rangle\langle 10| + |10\rangle\langle 11|$.

*Centre panel (Z error propagation):* A Z error (green $\pi$ spider) on the control qubit before a CNOT produces Z errors on **both** control and target after the CNOT. This follows from the Bialgebra rule (Rule 3). This is the classical result $\mathrm{CX}^\dagger (Z \otimes I) \mathrm{CX} = Z \otimes Z$ — but now stated as a graphical rewriting step. In the surface code this explains why Z stabilizer errors propagate along the control direction of each CX gate.

*Right panel (gate dictionary):* Every gate in `STIM_GATE_MAP` from `ising_surface_code_qec.ipynb` corresponds to a single Z or X spider with a specific phase. This means the entire `clifford_proxy_to_stim()` conversion is a translation between two representations of the same ZX diagram.

The X-error propagation rule is dual: an X error on the target of a CNOT produces X errors on **both** wires after the gate. The combination of Z-propagation and X-propagation is the foundation of how surface code stabilizers detect and localise errors.

---
## 5. The Ising Trotter Step in ZX

The one-step Ising Trotter circuit on two qubits reads:

$$\mathrm{CX}_{01} \;\to\; R_z(2J\Delta t)_1 \;\to\; \mathrm{CX}_{01} \;\to\; R_x(2h\Delta t)_0 \otimes R_x(2h\Delta t)_1$$

In ZX:
- Each CX is a Z-X spider pair.
- Each $R_z(\theta)$ is a Z spider with phase $\theta$.
- Each $R_x(\theta)$ is an X spider with phase $\theta$.

The full diagram can be simplified using spider fusion: the $R_z$ between two identical CX gates fuses with the inner Z spiders of each CX.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

def get_hamiltonian(n_qubits, J, h, alpha):
    ZZ = [('ZZ', [i, i+1], -J)              for i in range(n_qubits - 1)]
    Zf = [('Z',  [i],      -h*np.sin(alpha)) for i in range(n_qubits)]
    Xf = [('X',  [i],      -h*np.cos(alpha)) for i in range(n_qubits)]
    return SparsePauliOp.from_sparse_list(ZZ + Zf + Xf, num_qubits=n_qubits).simplify()

def build_trotter_ising(n_qubits, J, h, total_time, steps):
    qc = QuantumCircuit(n_qubits)
    dt = total_time / steps
    for _ in range(steps):
        for i in range(n_qubits - 1):
            qc.cx(i, i+1)
            qc.rz(2 * J * dt, i+1)
            qc.cx(i, i+1)
        for i in range(n_qubits):
            qc.rx(2 * h * dt, i)
        qc.barrier()
    return qc

# Show the 1-step, 2-qubit Ising Trotter circuit
qc_2q = build_trotter_ising(2, J, H_FIELD, TOTAL_TIME, 1)
print("One Trotter step, 2 qubits — circuit diagram:")
display(qc_2q.draw('text', fold=-1))

dt = TOTAL_TIME / 1
theta_zz = 2 * J * dt
theta_x  = 2 * H_FIELD * dt
print(f"\nPhase angles in the ZX diagram:")
print(f"  ZZ interaction:  θ_ZZ = 2J·Δt = {theta_zz:.4f} rad  (Z spider phase)")
print(f"  X-field:         θ_X  = 2h·Δt = {theta_x:.4f} rad  (X spider phase)")
print(f"  After Cliffordization: Z spiders replaced by random phase in {{0, π/2, π, 3π/2}}")

**Reading the Trotter circuit output.**  
The printed circuit shows the 2-qubit version of the 1-step Ising Trotter circuit from `ising_surface_code_qec.ipynb`. Reading left to right:

1. `CX` (qubits 0→1) — a Z-X spider pair in ZX.
2. `Rz(θ_ZZ)` on qubit 1 — a Z spider of phase $0.4$ rad.
3. `CX` (qubits 0→1) again — another Z-X spider pair.
4. `Rx(θ_X)` on each qubit — X spiders of phase $2.4$ rad.

The ZX diagram for this circuit can be simplified using spider fusion: the Z spider from the first CX's control, the $R_z$ spider, and the Z spider from the second CX's control all connect in sequence on qubit 0 and can be fused via Rule 1 into a single Z spider with phase $0 + \theta_{ZZ} + 0 = \theta_{ZZ}$.

After *Cliffordization*, $\theta_{ZZ}$ is replaced by a random Clifford phase $\in \{0, \pi/2, \pi, 3\pi/2\}$ and $\theta_X$ is replaced similarly. The ZX diagram becomes fully Clifford — all spider phases are multiples of $\pi/2$ — which is exactly what makes the circuit simulable by `stim`.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.set_xlim(-0.3, 13); ax.set_ylim(-1.6, 1.4)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title(
    'ZX Diagram — One Ising Trotter Step (2 qubits)\n'
    'Top wire: qubit 0 (control),  Bottom wire: qubit 1 (target + Rz)',
    fontsize=10
)

Z_COLOR = '#4CAF50'
X_COLOR = '#EF5350'
H_COLOR = '#FFEB3B'
θ_zz_str = f'{theta_zz:.2f}'
θ_x_str  = f'{theta_x:.2f}'

def seg(ax, x0, x1, y, **kw):
    ax.plot([x0, x1], [y, y], 'k-', lw=2, **kw)

def vseg(ax, x, y0, y1):
    ax.plot([x, x], [y0, y1], 'k-', lw=2)

def sp(ax, x, y, col, ph='', r=0.22):
    c = Circle((x, y), r, color=col, ec='black', lw=1.8, zorder=5)
    ax.add_patch(c)
    if ph:
        ax.text(x, y, ph, ha='center', va='center', fontsize=7,
                color='white', fontweight='bold', zorder=6)

# Input wires
seg(ax, 0.0, 1.0, 0.6); seg(ax, 0.0, 1.0, -0.8)
ax.text(-0.05, 0.6, 'q0', ha='right', fontsize=9, color='#555')
ax.text(-0.05, -0.8, 'q1', ha='right', fontsize=9, color='#555')

# CX gate 1: Z spider on q0, X spider on q1, vertical wire
seg(ax, 1.0, 1.8, 0.6); sp(ax, 2.0, 0.6, Z_COLOR)
vseg(ax, 2.0, 0.38, -0.58)
seg(ax, 1.0, 1.8, -0.8); sp(ax, 2.0, -0.8, X_COLOR)
seg(ax, 2.22, 3.0, 0.6); seg(ax, 2.22, 3.0, -0.8)
ax.text(1.1, 1.1, 'CX₁', fontsize=8, color='#333')

# Rz on q1
sp(ax, 3.2, -0.8, Z_COLOR, θ_zz_str, r=0.27)
seg(ax, 3.0, 3.0, 0.6)  # q0 passes through
seg(ax, 3.5, 4.3, -0.8)
seg(ax, 3.0, 4.3, 0.6)
ax.text(3.2, -1.28, f'Rz({θ_zz_str})', fontsize=7.5, ha='center', color='#4CAF50')

# CX gate 2
sp(ax, 4.5, 0.6, Z_COLOR); vseg(ax, 4.5, 0.38, -0.58); sp(ax, 4.5, -0.8, X_COLOR)
seg(ax, 4.3, 4.3, 0.6); seg(ax, 4.3, 4.3, -0.8)
seg(ax, 4.72, 5.5, 0.6); seg(ax, 4.72, 5.5, -0.8)
ax.text(4.6, 1.1, 'CX₂', fontsize=8, color='#333')

# Rx on each qubit
sp(ax, 5.7, 0.6, X_COLOR, θ_x_str, r=0.27)
sp(ax, 5.7, -0.8, X_COLOR, θ_x_str, r=0.27)
seg(ax, 5.5, 5.5, 0.6); seg(ax, 5.5, 5.5, -0.8)
seg(ax, 6.0, 7.0, 0.6); seg(ax, 6.0, 7.0, -0.8)
ax.text(5.7, 1.1, f'Rx({θ_x_str})', fontsize=7.5, ha='center', color='#EF5350')
ax.text(5.7, -1.28, f'Rx({θ_x_str})', fontsize=7.5, ha='center', color='#EF5350')

# Arrow to Cliffordized version
ax.annotate('', xy=(8.2, -0.1), xytext=(7.3, -0.1),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2.0))
ax.text(7.75, 0.2, 'Cliffordize\n(Rule 2 + fusion)', ha='center', fontsize=7.5, color='#333')

# Cliffordized version: same structure but phases from {0, π/2, π, 3π/2}
seg(ax, 8.2, 9.0, 0.6); seg(ax, 8.2, 9.0, -0.8)
sp(ax, 9.2, 0.6, Z_COLOR); vseg(ax, 9.2, 0.38, -0.58); sp(ax, 9.2, -0.8, X_COLOR)
seg(ax, 9.42, 10.2, 0.6); seg(ax, 9.42, 10.2, -0.8)
sp(ax, 10.4, -0.8, Z_COLOR, 'π/2', r=0.27)
seg(ax, 10.2, 11.2, 0.6); seg(ax, 10.7, 11.2, -0.8)
sp(ax, 11.4, 0.6, Z_COLOR); vseg(ax, 11.4, 0.38, -0.58); sp(ax, 11.4, -0.8, X_COLOR)
seg(ax, 11.62, 12.4, 0.6); seg(ax, 11.62, 12.4, -0.8)
sp(ax, 12.65, 0.6, X_COLOR, 'π', r=0.27)
sp(ax, 12.65, -0.8, X_COLOR, '0', r=0.27)

ax.text(10.5, 1.1, 'Random Clifford phases', fontsize=8, ha='center', color='#534AB7')

plt.tight_layout()
plt.show()

**Reading the Ising Trotter ZX diagram.**  
The left half of the figure is the full ZX diagram for one Ising Trotter step on 2 qubits. Follow the top wire (qubit 0) left to right: it enters the first CX as a Z spider (control), passes through unchanged past the $R_z$ on the other qubit, enters the second CX as a Z spider again, and finally receives the $R_x$ field gate as an X spider.

The connecting vertical wires between Z and X spiders at each CX gate are the "internal" wires of the ZX diagram — they carry no observable degrees of freedom but encode the entanglement structure. Crucially, the two CX gates are the only place where the two qubit wires interact, and they are *preserved exactly* by Cliffordization.

The right half shows the Cliffordized version: the non-Clifford phases $\theta_{ZZ}$ and $\theta_X$ are replaced by random multiples of $\pi/2$. In the example shown, the $R_z(\theta_{ZZ})$ has been replaced by an $S$ gate (phase $\pi/2$), and the $R_x$ on qubit 0 by an $X$ gate (phase $\pi$), while qubit 1's $R_x$ became the identity (phase $0$, eliminated by Rule 2). All phases are now in the Clifford set, so `stim` can simulate the resulting circuit.

**Key insight for the QEC pipeline:** The vertical connecting wires between CX spider pairs are identical in both diagrams. The Merkel et al. PTA theorem can be stated purely graphically: *any Clifford diagram with the same connectivity (internal wires) as the target has the same logical error rate under QEC.* ZX calculus makes this precise — the connectivity is the ZX graph structure, which Cliffordization preserves by definition.

---
## 6. Surface Code Stabilizers as Spiders

The surface code stabilizers from `7_surface_code_tutorial.ipynb` are directly representable as ZX spiders:

- A **Z-type plaquette stabilizer** $Z^{\otimes 4}$ (or $Z^{\otimes 2}$ at boundaries) is a **Z spider with 4 (or 2) output wires** and 0 input wires (acting as a measurement).
- An **X-type vertex stabilizer** $X^{\otimes 4}$ is an **X spider with 4 output wires**.
- The fact that all stabilizers commute with each other is the graphical statement that every Z-X pair of spiders that share data qubit wires do so on an *even* number of wires — by the Bialgebra rule, even overlaps cancel and the spiders commute.

### 6.1 Stabilizer measurement as ZX

Measuring a Z stabilizer $Z^{\otimes n}$ via an ancilla qubit is the following circuit:
1. Prepare ancilla in $|0\rangle$: a Z spider with 0 inputs, 1 output, phase 0.
2. Apply CNOT from each data qubit to the ancilla: a chain of Z-X spider pairs.
3. Measure ancilla in Z basis: a Z spider with 1 input, 0 outputs, phase 0.

The entire measurement gadget is a **Z spider with $n$ data-qubit inputs and 1 classical output wire** — Rule 1 (fusion) collapses the chain of CNOTs and the ancilla into a single multi-legged spider.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
Z_COLOR = '#4CAF50'
X_COLOR = '#EF5350'

# ── Left: d=3 stabilizer layout with ZX labels ───────────────────────────
ax = axes[0]
ax.set_xlim(-0.8, 2.8); ax.set_ylim(-0.8, 2.8)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('d=3 Stabilizers as Spiders\n(bulk only — boundary halves omitted)', fontsize=10)

d = 3
tile_colors = {'X': X_COLOR, 'Z': Z_COLOR}

# Tiles
for row in range(d - 1):
    for col in range(d - 1):
        ptype = 'X' if (row + col) % 2 == 0 else 'Z'
        cx_c, cy_c = col + 0.5, row + 0.5
        square = plt.Polygon(
            [(col, row), (col+1, row), (col+1, row+1), (col, row+1)],
            closed=True, color=tile_colors[ptype], alpha=0.25, zorder=0
        )
        ax.add_patch(square)
        # Spider at tile center
        c = Circle((cx_c, cy_c), 0.18, color=tile_colors[ptype], ec='black', lw=1.5, zorder=5)
        ax.add_patch(c)
        ax.text(cx_c, cy_c, ptype, ha='center', va='center',
                fontsize=7, color='white', fontweight='bold', zorder=6)
        # Legs to data qubits
        for dr, dc in [(-0.45, -0.45), (-0.45, 0.45), (0.45, -0.45), (0.45, 0.45)]:
            qr, qc = cy_c + dr, cx_c + dc
            if 0 <= qr <= d-1 and 0 <= qc <= d-1:
                ax.plot([cx_c, qc], [cy_c, qr], color='black', lw=1.2,
                        alpha=0.6, zorder=3)

# Data qubits
for r in range(d):
    for c in range(d):
        circ = Circle((c, r), 0.16, color='white', ec='#333', lw=1.8, zorder=7)
        ax.add_patch(circ)
        ax.text(c, r, 'D', ha='center', va='center', fontsize=7, color='#333', zorder=8)

# Legend
legend_elements = [
    mpatches.Patch(color=X_COLOR, alpha=0.5, label='X spider (X stabilizer)'),
    mpatches.Patch(color=Z_COLOR, alpha=0.5, label='Z spider (Z stabilizer)'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='white',
               markeredgecolor='#333', markersize=9, label='Data qubit (wire)'),
]
ax.legend(handles=legend_elements, loc='lower left', bbox_to_anchor=(0, -0.28),
          ncol=1, fontsize=8)

# ── Right: Z-stabilizer measurement gadget as ZX circuit ─────────────────
ax = axes[1]
ax.set_xlim(-0.5, 8.5); ax.set_ylim(-1.5, 3.5)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Z Stabilizer Measurement as ZX — before and after spider fusion', fontsize=10)

data_y = [3.0, 2.0, 1.0, 0.0]

# Ancilla prepare |0>
anc_prepare_x = 0.6
c_anc = Circle((anc_prepare_x, -0.7), 0.18, color=Z_COLOR, ec='black', lw=1.5, zorder=5)
ax.add_patch(c_anc)
ax.text(anc_prepare_x, -0.7, '0', ha='center', va='center', fontsize=7,
        color='white', fontweight='bold', zorder=6)
ax.text(anc_prepare_x - 0.05, -1.25, '|0⟩', fontsize=8, ha='center', color='#555')

# CNOTs (Z on data, X on ancilla)
cx_xs = [1.8, 2.8, 3.8, 4.8]
for i, (dy, cx_x) in enumerate(zip(data_y, cx_xs)):
    # Data wire
    ax.plot([-0.2, cx_x - 0.22], [dy, dy], 'k-', lw=1.8)
    sp_z = Circle((cx_x, dy), 0.2, color=Z_COLOR, ec='black', lw=1.5, zorder=5)
    ax.add_patch(sp_z)
    ax.plot([cx_x + 0.2, 6.0], [dy, dy], 'k-', lw=1.8)
    # Ancilla wire segment
    anc_y = -0.7
    ax.plot([anc_prepare_x if i == 0 else cx_xs[i-1],
             cx_x], [anc_y, anc_y], 'k-', lw=1.8)
    sp_x = Circle((cx_x, anc_y), 0.2, color=X_COLOR, ec='black', lw=1.5, zorder=5)
    ax.add_patch(sp_x)
    ax.plot([cx_x, cx_x], [dy - 0.2, anc_y + 0.2], 'k-', lw=1.5, alpha=0.7)

# Ancilla measure
ax.plot([cx_xs[-1], 5.5], [-0.7, -0.7], 'k-', lw=1.8)
sp_meas = Circle((5.7, -0.7), 0.18, color=Z_COLOR, ec='black', lw=1.5, zorder=5)
ax.add_patch(sp_meas)
ax.text(5.7, -0.7, 'M', ha='center', va='center', fontsize=7,
        color='white', fontweight='bold', zorder=6)
ax.text(6.0, -0.7, '→ syndrome bit', fontsize=8, va='center', color='#333')

# Arrow to fused version
ax.annotate('', xy=(6.2, 1.2), xytext=(5.7, 1.2),
            xycoords='data', textcoords='data',
            arrowprops=dict(arrowstyle='->', color='#334', lw=2))
ax.text(5.1, 1.5, 'Fusion\n(Rule 1)', fontsize=7.5, ha='right', color='#333')

# Fused: single Z spider with 4 legs + measurement output
fused_x = 7.5
fused_y = 1.2
big_sp = Circle((fused_x, fused_y), 0.35, color=Z_COLOR, ec='black', lw=2.0, zorder=5)
ax.add_patch(big_sp)
ax.text(fused_x, fused_y, 'Z', ha='center', va='center',
        fontsize=10, color='white', fontweight='bold', zorder=6)
for dy in data_y:
    ax.plot([6.0, fused_x - 0.35], [dy, fused_y + (dy - fused_y) * 0.5 / abs(dy - fused_y + 1e-9)],
            'k--', lw=1.2, alpha=0.5)
    ax.plot([6.0, fused_x - 0.3], [dy, fused_y], 'k-', lw=1.5, alpha=0.6)

ax.text(fused_x + 0.38, fused_y, '→ s', fontsize=9, va='center', color='#333')

plt.suptitle('Surface Code Stabilizers in ZX Calculus', fontsize=12, y=1.0)
plt.tight_layout()
plt.show()

**Reading the stabilizer figures.**  

*Left panel — stabilizer layout:* The $d=3$ rotated surface code from `7_surface_code_tutorial.ipynb` is redrawn with each stabilizer ancilla replaced by its corresponding ZX spider. Green circles at tile centres are Z spiders (measuring $Z^{\otimes 4}$), red circles are X spiders (measuring $X^{\otimes 4}$). The legs of each spider connect to the surrounding data qubit wires. The fact that Z and X spiders *alternate* on the checkerboard is why the stabilizers commute — every Z-X pair shares either 0 or 2 data qubits (even overlap → Bialgebra rule gives trivial commutation).

*Right panel — measurement gadget:* The upper half shows the explicit circuit for measuring a 4-body Z stabilizer: prepare an ancilla in $|0\rangle$ (a Z spider cap), apply four CNOT gates from each data qubit to the ancilla (four Z-X spider pairs with vertical connecting wires), then measure the ancilla (a Z spider cup). By the Fusion rule (Rule 1), all four CNOT Z spiders on the control side fuse with the ancilla preparation Z spider and the measurement Z spider into a **single large Z spider with 4 data-qubit wires and 1 classical output wire** — the lower-right diagram. This single spider *is* the $Z^{\otimes 4}$ stabilizer measurement gadget.

---
## 7. Syndrome Extraction Circuits in ZX

A full QEC round in `stim` consists of:
1. Reset all ancillas: $|0\rangle$ preparation = Z spider caps.
2. Hadamard X-type ancillas: H boxes on X-spider wires.
3. Four CNOT gates per ancilla in a fixed order.
4. Hadamard X-type ancillas again.
5. Measure all ancillas: Z spider cups.

We can verify the entire stim-generated syndrome extraction circuit here and show how the ZX simplification (fusion) confirms the detectors are correctly formed.

In [ ]:
# Generate the d=3 surface code circuit and inspect its structure
d = CODE_DIST
p = P_PHYS

circ = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=d,
    distance=d,
    after_clifford_depolarization=p,
    before_round_data_depolarization=0.0,
    after_reset_flip_probability=0.0,
    before_measure_flip_probability=0.0,
)

print(f"d={d} surface code memory circuit — ZX-relevant statistics:")
print(f"  Qubits      : {circ.num_qubits}   = 2d²-1 = {2*d**2-1}")
print(f"  Ancillas    : {d**2 - 1}   = d²-1")
print(f"  Data qubits : {d**2}  = d²")
print()

# Count instruction types
gate_counts = {}
for inst in circ.flattened():
    name = inst.name
    gate_counts[name] = gate_counts.get(name, 0) + 1

print("Gate counts in the full circuit:")
for k, v in sorted(gate_counts.items(), key=lambda x: -x[1]):
    zx_note = {
        'CX':   '← Z-X spider pairs (main entanglement)         ',
        'H':    '← Hadamard boxes (Z↔X colour change)           ',
        'R':    '← Z spider cap (ancilla reset to |0⟩)          ',
        'MR':   '← Z spider cup+cap (measure then reset)        ',
        'M':    '← Z spider cup (final measurement)             ',
        'TICK': '← time-layer separator                         ',
    }.get(k, '')
    print(f"  {k:<18}: {v:>5}  {zx_note}")

**Reading the circuit statistics.**  
Every instruction in the `stim` circuit corresponds to a ZX primitive:

- **`R` (reset):** A Z spider cap — a cup with 0 inputs and 1 output, phase 0 — preparing $|0\rangle$. In ZX terms this is a "state-preparation" node.
- **`H` (Hadamard):** An H box converting Z wires to X wires. Applied before and after X-type ancilla measurement rounds to convert the X-basis measurement into a Z-basis one.
- **`CX`:** The Z-X spider pair. There are $4 \times (d^2 - 1) \times d$ CX gates in total: 4 per ancilla per round, $(d^2-1)$ ancillas, $d$ rounds.
- **`MR` (measure-reset):** A Z spider cup (measurement) immediately followed by a Z spider cap (re-initialisation) at the end of each round. The measurement result feeds a detector.
- **`M` (final measure):** Z spider cups on data qubits at the end of the experiment. These results, combined with the ancilla history, compute the logical observable $\bar{Z}$.
- **`TICK`:** A separator between time layers. In ZX, all gates within a `TICK` block are applied simultaneously — they act on independent wires and their Z-X pairs do not interact within the same `TICK`.

The ratio of CX gates to ancillas is approximately 4 — each ancilla has 4 CNOT connections (weight-4 stabilizers). Boundary ancillas have 2 connections (weight-2 stabilizers), reducing the average slightly below 4.

In [ ]:
# Verify ZX fusion claim: ancilla prepare → CNOT chain → measure = Z(⊗n) stabilizer
# Build the minimal 2-round stabilizer circuit for one Z-type ancilla
# and confirm the DEM entries match the expected detector structure

mini_circ = stim.Circuit("""
    # Data qubits: 0,1,2,3   Ancilla: 4
    # Round 1: Z stabilizer measurement on all 4 data qubits
    R 4
    TICK
    CX 0 4
    TICK
    CX 1 4
    TICK
    CX 2 4
    TICK
    CX 3 4
    TICK
    MR 4
    # Detector: round-1 ancilla (no previous round to XOR with, so referenced alone)
    DETECTOR rec[-1]

    # Round 2: same measurement (compare to round 1 for error detection)
    TICK
    CX 0 4
    TICK
    CX 1 4
    TICK
    CX 2 4
    TICK
    CX 3 4
    TICK
    MR 4
    # Detector: XOR of round-2 and round-1 (catches errors between rounds)
    DETECTOR rec[-1] rec[-2]

    # Final: measure data qubits
    M 0 1 2 3
    # Observable: logical Z = all data qubits in Z basis
    OBSERVABLE_INCLUDE(0) rec[-1] rec[-2] rec[-3] rec[-4]
""")

print("Minimal 2-round Z-stabilizer circuit:")
print(f"  Qubits      : {mini_circ.num_qubits}  (4 data + 1 ancilla)")
print(f"  Measurements: {mini_circ.num_measurements}  (2 ancilla rounds + 4 final)")
print(f"  Detectors   : {mini_circ.num_detectors}")
print()
print("ZX interpretation:")
print("  Each round: R(ancilla) → CX×4 → MR(ancilla)")
print("            = Z-cap → [Z-X pairs × 4] → Z-cup")
print("            = (by fusion) single Z spider with 4 data-qubit legs")
print("  DETECTOR = XOR of two consecutive Z-spider measurement outcomes")
print("           = fires iff the Z-stabilizer value CHANGED between rounds")
print("           = ZX diagram: two Z-cups connected by XOR (classical post-processing)")

**Reading the minimal Z-stabilizer circuit.**  
This circuit implements exactly one Z-type stabilizer, repeated for 2 rounds, with the detector defined as the XOR of the two consecutive ancilla measurements.

The ZX interpretation (printed at the bottom) shows how spider fusion collapses the entire measurement gadget:
1. `R 4` is a Z spider cap: $|0\rangle$ preparation, a node with 0 inputs and 1 output.
2. Each `CX i 4` is a Z-X spider pair. The Z spider on qubit `i` (control) and the X spider on qubit 4 (ancilla target) share an internal wire.
3. By Rule 1 (Fusion), all four X spiders on the ancilla wire fuse with each other and with the ancilla Z-cap into a single X spider with 4 data-qubit wires.
4. By Rule 5 (Colour Change, two H boxes), the X spider at the ancilla is converted back to Z: the sequence H-X-H = Z. This is the ZX explanation of why the CNOT-then-measure circuit measures the Z stabilizer.
5. `MR 4` is a Z spider cup — measuring the ancilla in the Z basis — immediately followed by another Z spider cap for the next round.
6. The `DETECTOR` is classical: XOR the two measurement outcomes. A ZX diagram of classical post-processing uses grey wires, but the stim language handles this implicitly.

---
## 8. Pauli Propagation and Cliffordization via ZX

Sections 4 and 5 of `ising_surface_code_qec.ipynb` introduced two operations — **Pauli twirling** and **Cliffordization** — that can now be understood as ZX rewriting.

### 8.1 Pauli twirling as ZX (Rule 4)

Pauli twirling wraps every CX gate in random Pauli gates:
$$P_\text{before} \cdot \mathrm{CX} \cdot P_\text{after}$$

In ZX, a Pauli gate is a phase-$\pi$ spider. The π-copy rule (Rule 4) says:
$$X(\pi) \circ Z(0) = Z(0) \circ X(\pi) \quad\text{(X passes through Z unchanged)}$$
$$Z(\pi) \circ X(0) = X(0) \circ Z(\pi) \quad\text{(Z passes through X unchanged)}$$
$$X(\pi) \circ Z(\alpha) = Z(-\alpha) \circ X(\pi) \quad\text{(X conjugates Z phase)}$$

These are the commutation relations between Paulis and rotation gates. Twirling randomises the phases, and by the ZX rules, the average over random Pauli wrappings converts any coherent Z(α) phase error into a **stochastic Pauli channel** — the ZX diagram for the averaged circuit is an incoherent mixture of Z(0) and Z(π), i.e., a depolarizing channel.

### 8.2 Cliffordization as ZX (Rules 1 + 2)

Cliffordization replaces each non-Clifford Z(α) spider with a random Z(kπ/2) spider ($k \in \{0,1,2,3\}$). In ZX:
- The circuit graph topology is unchanged (same wires, same connectivity).
- Only the phase labels on the single-qubit spiders change.
- All ZX rewriting rules (Fusion, Identity, Bialgebra) are phase-label manipulations — they apply equally to Clifford and non-Clifford diagrams.

Therefore: **a ZX simplification that works on the Clifford proxy also works on the original Ising circuit, and vice versa.** This is the ZX-calculus formulation of the PTA theorem.

In [ ]:
# Demonstrate Rule 4 numerically: X(π) commutes through Z(α) with phase negation

def rx_mat(theta):
    """Single-qubit Rx(theta) matrix."""
    c, s = np.cos(theta/2), np.sin(theta/2)
    return np.array([[c, -1j*s], [-1j*s, c]])

def rz_mat(theta):
    """Single-qubit Rz(theta) matrix."""
    return np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]])

X_gate = np.array([[0, 1], [1, 0]])
Z_gate = np.array([[1, 0], [0, -1]])

print("Pauli propagation rules — numerical verification")
print("=" * 55)

for alpha in [np.pi/4, np.pi/3, 0.7, np.pi/2]:
    # Rule 4: X(π) ∘ Rz(α) = Rz(-α) ∘ X(π)
    LHS = X_gate @ rz_mat(alpha)
    RHS = rz_mat(-alpha) @ X_gate
    match_4 = np.allclose(LHS, RHS)

    # Colour-change dual: Z(π) ∘ Rx(α) = Rx(-α) ∘ Z(π)
    LHS2 = Z_gate @ rx_mat(alpha)
    RHS2 = rx_mat(-alpha) @ Z_gate
    match_5 = np.allclose(LHS2, RHS2)

    print(f"  α = {alpha:.4f}:  X·Rz(α) = Rz(-α)·X: {match_4}  |  "
          f"Z·Rx(α) = Rx(-α)·Z: {match_5}")

print()
print("Cliffordization consistency check:")
print("  For Clifford phases {0, π/2, π, 3π/2}: -α mod 2π stays in the set.")
for alpha_c in [0, np.pi/2, np.pi, 3*np.pi/2]:
    neg_alpha = (-alpha_c) % (2 * np.pi)
    is_clifford = any(np.isclose(neg_alpha, k * np.pi / 2) for k in range(4))
    print(f"  -({alpha_c:.3f}) mod 2π = {neg_alpha:.3f}   Clifford: {is_clifford}")

**Reading the Pauli propagation verification.**  
Each row confirms the π-copy rule (Rule 4) numerically for a different rotation angle $\alpha$: the matrices $X \cdot R_z(\alpha)$ and $R_z(-\alpha) \cdot X$ are identical, as are $Z \cdot R_x(\alpha)$ and $R_x(-\alpha) \cdot Z$. These are not approximations — they are exact matrix equalities.

The Cliffordization consistency check at the bottom is a key bookkeeping result: negating a Clifford phase (a multiple of $\pi/2$) produces another Clifford phase. This means that when Pauli twirling is applied to a Cliffordized circuit, the resulting phases are still Clifford. The proxy circuits from `ising_surface_code_qec.ipynb` remain in the stim-simulable Clifford subtheory even after arbitrary Pauli conjugation — this is the ZX proof that twirling and Cliffordization commute.

For non-Clifford phases (like $\alpha = \pi/4$ in the true Ising circuit), $-\alpha = -\pi/4 = 7\pi/4$, which is *not* in the Clifford set. This is why the real Ising circuit cannot be directly simulated by `stim`: Pauli conjugation moves it outside the Clifford subtheory. The Clifford proxy stays inside — that is precisely what makes it useful.

---
## 9. Lattice Surgery as ZX Composition

Section 6 of `ising_surface_code_qec.ipynb` describes how a logical CX between two surface code patches is performed via lattice surgery: a merge phase (measuring $\bar{X}_c \otimes \bar{X}_t$) followed by a split phase. In ZX calculus, this is the *composition* of two large spiders along a shared boundary.

### 9.1 Logical operators as ZX diagrams

The logical operators of the $d=3$ surface code are:
- $\bar{X}$: a row of X operators across the patch = an X spider with 3 data-qubit legs (horizontal string).
- $\bar{Z}$: a column of Z operators down the patch = a Z spider with 3 data-qubit legs (vertical string).

Both are minimum-weight strings of length $d=3$ — this is why a single error of weight $< d$ cannot mimic them.

### 9.2 Logical CX via merge and split

The merge phase measures the **joint** stabilizer $\bar{X}_c \otimes \bar{X}_t$ on two patches. In ZX:

$$\text{merge} = \text{X spider (control } \bar{X}\text{)} \xrightarrow{\text{fuse across boundary}} \text{X spider (target } \bar{X}\text{)}$$

Joining the two $\bar{X}$ strings along a shared boundary fuses them into a single X spider with 6 data-qubit legs spanning both patches. Measuring this joint X spider yields one bit of syndrome information — the parity of the logical XX operator — which, after a classical correction step, implements the logical CX gate.

The split phase is the time-reversal of the merge: the joint spider is broken back into two single-patch logical operators. In ZX, this is the reverse of fusion — a spider with $n+m$ legs is written as a composition of two spiders with $n$ and $m$ legs connected by an internal wire.

This shows that the entire lattice surgery protocol for a logical CX is a single application of the ZX Spider Fusion rule (Rule 1) followed by its inverse — a conceptually simple operation, even though implementing it on hardware requires $2d$ rounds of syndrome extraction.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
Z_COLOR = '#4CAF50'
X_COLOR = '#EF5350'

for ax in axes:
    ax.set_aspect('equal'); ax.axis('off')

def draw_patch(ax, xoff, yoff, d=3, label='', color=Z_COLOR, show_logical='Z'):
    """Draw a simplified surface code patch as a grid of data qubits."""
    for r in range(d):
        for c in range(d):
            circ = Circle((xoff + c * 0.5, yoff + r * 0.5), 0.12,
                           color='white', ec='#444', lw=1.4, zorder=5)
            ax.add_patch(circ)
    # Logical operator
    if show_logical == 'Z':
        mid = d // 2
        for r in range(d):
            c = Circle((xoff + mid * 0.5, yoff + r * 0.5), 0.14,
                        color=Z_COLOR, ec='black', lw=1.2, alpha=0.7, zorder=6)
            ax.add_patch(c)
        ax.text(xoff + mid * 0.5, yoff + d * 0.5 + 0.1, r'$\bar{Z}$',
                ha='center', fontsize=9, color=Z_COLOR)
    elif show_logical == 'X':
        mid = d // 2
        for c in range(d):
            circ = Circle((xoff + c * 0.5, yoff + mid * 0.5), 0.14,
                           color=X_COLOR, ec='black', lw=1.2, alpha=0.7, zorder=6)
            ax.add_patch(circ)
        ax.text(xoff + d * 0.5, yoff + mid * 0.5, r'$\bar{X}$',
                ha='left', fontsize=9, color=X_COLOR)
    box = plt.Polygon(
        [(xoff - 0.15, yoff - 0.15), (xoff + (d-1)*0.5 + 0.15, yoff - 0.15),
         (xoff + (d-1)*0.5 + 0.15, yoff + (d-1)*0.5 + 0.15), (xoff - 0.15, yoff + (d-1)*0.5 + 0.15)],
        fill=False, ec='#888', lw=1.2, ls='--', zorder=1
    )
    ax.add_patch(box)
    ax.text(xoff + (d-1)*0.5/2, yoff - 0.35, label, ha='center', fontsize=9, color='#333')

# ── Panel 1: Two separate patches ────────────────────────────────────────
ax = axes[0]; ax.set_xlim(-0.3, 4.5); ax.set_ylim(-0.8, 2.0)
ax.set_title('Before merge:\ntwo logical qubits', fontsize=9)
draw_patch(ax, 0.0, 0.0, label='Control (c)', show_logical='X')
draw_patch(ax, 2.3, 0.0, label='Target (t)', show_logical='X')
# Two separate X-bar strings
ax.text(2.0, 0.55, '|', ha='center', va='center', fontsize=18, color='#aaa')

# ── Panel 2: Merge — joint X_c X_t measured ──────────────────────────────
ax = axes[1]; ax.set_xlim(-0.3, 4.5); ax.set_ylim(-0.8, 2.0)
ax.set_title('Merge phase (d rounds):\njoint X̄_c ⊗ X̄_t measured', fontsize=9)
draw_patch(ax, 0.0, 0.0, label='Control (c)', show_logical='X')
draw_patch(ax, 2.3, 0.0, label='Target (t)', show_logical='X')
# Shared boundary region highlighted
merge_rect = plt.Polygon(
    [(1.85, -0.15), (2.45, -0.15), (2.45, 1.15), (1.85, 1.15)],
    facecolor='#FFD54F', alpha=0.4, edgecolor='#FF8F00', lw=1.5, zorder=0
)
ax.add_patch(merge_rect)
ax.text(2.15, -0.55, 'boundary\nancillas', ha='center', fontsize=7, color='#8B5E00')
# Joint X spider spanning both
big = Circle((2.15, 0.5), 0.25, color=X_COLOR, ec='black', lw=1.8, alpha=0.9, zorder=8)
ax.add_patch(big)
ax.text(2.15, 0.5, 'XX', ha='center', va='center', fontsize=7.5,
        color='white', fontweight='bold', zorder=9)
# Legs from big spider to both patches
for dy in [0.0, 0.5, 1.0]:
    ax.plot([1.1, 1.9], [dy, 0.5], X_COLOR, lw=1.0, alpha=0.5, ls='--')
    ax.plot([2.4, 2.3], [0.5, dy], X_COLOR, lw=1.0, alpha=0.5, ls='--')

# ── Panel 3: ZX diagram for logical CX ───────────────────────────────────
ax = axes[2]; ax.set_xlim(-0.5, 5.0); ax.set_ylim(-1.0, 2.0)
ax.set_title('ZX diagram of logical CX\n(merge + split = Rule 1 forward + reverse)', fontsize=9)

# Control wire
ax.plot([-0.3, 0.7], [1.2, 1.2], 'k-', lw=2)
ctrl_sp = Circle((0.9, 1.2), 0.22, color=Z_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(ctrl_sp)
ax.text(0.9, 1.2, 'Z̄', ha='center', va='center', fontsize=8,
        color='white', fontweight='bold', zorder=6)
ax.plot([1.12, 4.3], [1.2, 1.2], 'k-', lw=2)

# Internal wire (merge+split)
ax.plot([1.12, 2.5], [1.2, 0.4], 'k-', lw=1.5, alpha=0.7)
ax.text(1.8, 0.85, 'merge', fontsize=7.5, rotation=30, color='#FF8F00')
ax.text(2.9, 0.85, 'split', fontsize=7.5, rotation=-25, color='#1565C0')

# Target wire
ax.plot([-0.3, 0.7], [0.0, 0.0], 'k-', lw=2)
tgt_sp = Circle((0.9, 0.0), 0.22, color=X_COLOR, ec='black', lw=1.8, zorder=5)
ax.add_patch(tgt_sp)
ax.text(0.9, 0.0, 'X̄', ha='center', va='center', fontsize=8,
        color='white', fontweight='bold', zorder=6)
ax.plot([1.12, 4.3], [0.0, 0.0], 'k-', lw=2)

# Joint measurement spider (the merge)
joint = Circle((2.5, 0.4), 0.28, color=X_COLOR, ec='black', lw=1.8, alpha=0.85, zorder=7)
ax.add_patch(joint)
ax.text(2.5, 0.4, 'X̄cX̄t', ha='center', va='center', fontsize=6.5,
        color='white', fontweight='bold', zorder=8)
ax.plot([2.5, 2.5], [0.68, 1.2 - 0.22], 'k-', lw=1.5, alpha=0.7)
ax.plot([2.5, 2.5], [0.12, 0.0 + 0.22], 'k-', lw=1.5, alpha=0.7)
ax.plot([2.78, 3.5], [0.4, 0.4], 'k-', lw=1.5, ls='--', alpha=0.5)
ax.text(3.6, 0.4, '→ classical bit', fontsize=7.5, va='center', color='#555')

ax.text(-0.35, 1.2, 'ctrl', ha='right', fontsize=8, color='#333')
ax.text(-0.35, 0.0, 'tgt',  ha='right', fontsize=8, color='#333')

plt.suptitle('Lattice Surgery as ZX Spider Fusion (Rule 1)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**Reading the lattice surgery ZX figure.**  

*Left panel — before merge:* Two separate surface code patches, each with its own $\bar{X}$ logical string (red dots). The vertical bar between them indicates a physical boundary — the two patches are not yet connected.

*Centre panel — merge phase:* During the $d$ rounds of merge, additional ancilla qubits at the shared boundary measure the joint $\bar{X}_c \otimes \bar{X}_t$ stabilizer. In ZX, this is the joint X spider at the boundary (the orange XX node) with legs connecting into both patches. The spider "fuses" the two $\bar{X}$ strings into one long string spanning both patches — Rule 1 in action.

*Right panel — ZX diagram of logical CX:* The full logical CX circuit is represented as two logical qubit wires (control = green Z-spider wire, target = red X-spider wire) with an internal merge/split gadget (the orange joint X-spider). The dashed wire out of the joint spider represents the classical measurement result — one bit that, after correction, implements the logical gate. The split is the time-reverse of the merge: the joint spider is separated back into two single-patch spiders, completing the gate.

This ZX diagram is exactly the `make_logical_cx_circuit()` circuit from `ising_surface_code_qec.ipynb`, expressed graphically. The 2-round ancilla measurement in that code (one detector) corresponds to the merge phase; the final data qubit measurements (observable) correspond to the split.

---
## 10. Discussion and Conclusions

### 10.1 What ZX calculus adds to the pipeline

The three prior notebooks built a complete QEC pipeline:

```
Ising Hamiltonian
  → Trotter circuit (non-Clifford)
  → Cliffordization + PTA (Merkel et al.)
  → stim.Circuit (Clifford proxy)
  → Surface code encoding
  → Syndrome extraction + MWPM
  → Logical error rate comparison
```

ZX calculus provides the **semantic layer** underneath this pipeline:

| Pipeline step | ZX foundation |
|---|---|
| Trotter circuit | Z and X spiders with continuous phases |
| Cliffordization | Restrict spider phases to $k\pi/2$ |
| PTA theorem | Phase-invariance of the ZX graph structure |
| `STIM_GATE_MAP` translation | Spider phase lookup table |
| Stabilizer generators | Multi-legged spiders (via fusion) |
| Syndrome extraction circuit | Spider cap → chain → cup |
| Error propagation (Pauli twirling) | π-copy rule (Rule 4) |
| Lattice surgery | Fusion rule (Rule 1) |

### 10.2 ZX for circuit optimisation

The `pyzx` library implements `full_reduce()` — a systematic application of ZX rewriting rules that provably simplifies any Clifford circuit to a near-minimal form. Applied to the Clifford proxy circuits from Section 5 of `ising_surface_code_qec.ipynb`, `full_reduce()` can eliminate redundant single-qubit gates (those that fuse to identity via Rule 2) and merge adjacent rotation spiders (Rule 1), potentially reducing the logical CX count and therefore the QEC round budget.

### 10.3 ZX for formal QEC verification

The ZX completeness theorem (Jeandel, Perdrix, Vilmart 2018) guarantees that if two diagrams compute the same linear map, they are connected by ZX rewrites. This means that a ZX-based proof system can, in principle, formally verify that a given stim circuit implements the correct syndrome extraction for a given stabilizer code — without resorting to matrix multiplication.

### 10.4 Recommended next steps

1. **Install `pyzx`** and apply `pyzx.full_reduce()` to the 1-step Clifford proxy circuits from `ising_surface_code_qec.ipynb`; compare the gate counts before and after.
2. **ZX for T gates:** Extend the framework to non-Clifford phases by replacing Z($\pi/4$) spiders with the T-gate magic state gadget — the starting point for universal fault-tolerant computation.
3. **ZX for decoder verification:** Draw the MWPM matching graph as a ZX diagram and verify that matched correction operators cancel the detected errors.
4. **Scalability:** Repeat the analysis for $N = 8$ or $N = 16$ Ising qubits; observe how the ZX graph grows and which simplifications `full_reduce()` finds.

---

### References

- **Coecke & Duncan (2008, 2011)** — Original ZX calculus papers.
- **Jeandel, Perdrix, Vilmart (2018)** — Completeness of ZX calculus for Clifford+T.
- **van de Wetering (2020)** — *ZX-Calculus for the Working Quantum Computer Scientist* (arXiv:2012.13966). Highly recommended.
- **Kissinger & van de Wetering (2019)** — `pyzx` library (arXiv:1904.04735).
- **Merkel et al. (2025)** — Cliffordization and PTA theorem (cited in `ising_surface_code_qec.ipynb`).
- **Gidney (2021)** — `stim` stabilizer simulator.
- **Fowler et al. (2012)** — Surface code review, threshold analysis.